# BERTurk denemesi (A0.2)

Amaç: `dbmdz/bert-base-turkish-cased` modelinin Colab'da yüklenip çalıştığını görmek ve A2.1'deki
çok etiketli kurulumu (`aspect:polarity`) küçük bir örnekte denemek.

- Colab: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** (CPU'da da çalışır, yavaştır).
- Cümleler **uydurmadır**. Gerçek yorum metni bu deftere ve repoya girmez.
- Buradaki eğitim yalnızca kurulumun çalıştığını gösterir; bir sonuç değildir.

In [ ]:
!pip -q install "transformers>=4.40" "torch>=2.2"

In [ ]:
import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

MODEL = "dbmdz/bert-base-turkish-cased"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print("cihaz:", device, "| sözlük boyutu:", tokenizer.vocab_size)

## 1. Türkçe metin nasıl parçalanıyor?

Eklemeli yapı yüzünden uzun kelimeler alt parçalara bölünür (`##` ile başlayanlar). Yazım hatası ve
uzatılmış harf ("çoooook") daha çok parçaya bölünür; temizlemede harf uzatmayı kısaltmamızın bir nedeni bu.

In [ ]:
for s in ["Makine çok sessiz çalışıyor.",
          "Dayanıklılığından memnun değilim, iki ayda arızalandı.",
          "çoooook güzel ama servis gelmedi",
          "Elektrik faturamız düştü."]:
    print(s, "->", tokenizer.tokenize(s))

## 2. Model yükleniyor ve cümle temsili çıkıyor

`[CLS]` vektörü 768 boyutludur. Benzer anlamlı cümlelerin benzerliği daha yüksek çıkmalı, ama
eğitilmemiş BERT'te bu her zaman net olmaz; sınıflandırma için fine-tune gerekmesinin nedeni bu.

In [ ]:
encoder = AutoModel.from_pretrained(MODEL).to(device).eval()

def cls_vector(text):
    batch = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        return encoder(**batch).last_hidden_state[0, 0]

a, b, c = (cls_vector(t) for t in ["Çok sessiz çalışıyor.", "Sesi hiç duyulmuyor.",
                                    "Kargo geç geldi."])
cos = torch.nn.functional.cosine_similarity
print("boyut:", a.shape[0])
print("sessiz ~ sesi duyulmuyor:", round(cos(a, b, dim=0).item(), 3))
print("sessiz ~ kargo geç      :", round(cos(a, c, dim=0).item(), 3))

## 3. Çok etiketli sınıflandırma kurulumu (A2.1'in iskeleti)

Etiketler `config/aspects.yaml` ile aynı: 5 aspect × olumlu/olumsuz, `failure:pos` yok → 9 etiket.
Bir yorumda birden fazla etiket olabilir (ör. arıza + servis), bu yüzden softmax değil sigmoid
(`problem_type="multi_label_classification"`, kayıp: BCE).

In [ ]:
LABELS = ["durability:pos", "durability:neg", "energy:pos", "energy:neg", "service:pos",
          "service:neg", "failure:neg", "noise:pos", "noise:neg"]

# Uydurma örnekler: (metin, etiketler)
examples = [
    ("Üç yıldır sorunsuz çalışıyor, çok sessiz.", ["durability:pos", "noise:pos"]),
    ("İki ayda arızalandı, servis de gelmedi.", ["failure:neg", "service:neg"]),
    ("Elektrik faturamız belirgin şekilde düştü.", ["energy:pos"]),
    ("Sıkmada çok ses yapıyor ve titriyor.", ["noise:neg"]),
    ("Servis aynı gün gelip kurdu, çok ilgiliydiler.", ["service:pos"]),
    ("Kargo hızlıydı, paketleme güzeldi.", []),
    ("Kapak menteşesi ilk haftada kırıldı.", ["durability:neg"]),
    ("Çok fazla elektrik harcıyor.", ["energy:neg"]),
]

def target(labels):
    return [1.0 if name in labels else 0.0 for name in LABELS]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), problem_type="multi_label_classification",
    id2label=dict(enumerate(LABELS)), label2id={n: i for i, n in enumerate(LABELS)},
).to(device)

batch = tokenizer([t for t, _ in examples], padding=True, truncation=True, max_length=128,
                  return_tensors="pt").to(device)
y = torch.tensor([target(l) for _, l in examples], device=device)
print("girdi:", tuple(batch["input_ids"].shape), "| hedef:", tuple(y.shape))

## 4. Küçük deneme eğitimi

8 örnek üzerinde birkaç adım: kaybın düşmesi kurulumun doğru olduğunu gösterir. Model bu örnekleri
ezberler; genelleme hakkında **hiçbir şey söylemez**. Gerçek eğitim etiketli train setiyle, sonuç
yalnızca test setiyle raporlanır.

In [ ]:
torch.manual_seed(491)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()
for step in range(1, 31):
    loss = model(**batch, labels=y).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step % 5 == 0:
        print(f"adım {step:2d}  kayıp {loss.item():.4f}")

In [ ]:
model.eval()
THRESHOLD = 0.5  # A2.1'de doğrulama setinde ayarlanacak

def predict(texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128,
                    return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits)
    return [[LABELS[j] for j, p in enumerate(row) if p >= THRESHOLD] for row in probs]

# Eğitimde olmayan uydurma cümleler: sonuçlar rastgele olabilir, yalnızca akışı gösterir
for text, labels in zip(["Hiç ses çıkarmıyor.", "Servis bir haftadır gelmiyor."],
                        predict(["Hiç ses çıkarmıyor.", "Servis bir haftadır gelmiyor."])):
    print(text, "->", labels)

## Not edilecekler (docs/literatur.md'ye)

- Colab'da GPU türü, model indirme süresi, 30 adımın süresi.
- Hata çıktıysa mesajın kendisi.